# 01 — Data Understanding

Dataset overview, schema, data quality, and preparation decisions.

## 1. Project objectives

Understand the UCI Online Retail transactions and establish data quality considerations before analysis.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = Path.cwd().parent

REPORTS = ROOT / "reports"
PROCESSED = ROOT / "data" / "processed"
FIGURES = REPORTS / "figures"

print("Project root:", ROOT)
print("Reports directory exists:", REPORTS.exists())
print("Processed data directory exists:", PROCESSED.exists())

Project root: C:\Users\tadij\Documents\retail-customer-intelligence-python
Reports directory exists: True
Processed data directory exists: True


## 2. Dataset dimensions and schema

In [2]:
data_path = PROCESSED / "online_retail_prepared.csv"
df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())
display(df.dtypes.to_frame("data_type"))

Rows: 541909
Columns: 10


,data_type
invoiceno,str
stockcode,str
description,str
quantity,int64
invoicedate,str
unitprice,float64
customerid,float64
country,str
is_cancellation,bool
line_value,float64


## 3. Missing values and duplicates

In [3]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
display(quality.sort_values("missing_count", ascending=False))
print("Exact duplicate rows:", int(df.duplicated().sum()))

,missing_count,missing_percent
customerid,135080,24.93
description,1454,0.27
stockcode,0,0.00
invoiceno,0,0.00
invoicedate,0,0.00
quantity,0,0.00
unitprice,0,0.00
country,0,0.00
is_cancellation,0,0.00
line_value,0,0.00


Exact duplicate rows: 5268


## 4. Transaction anomalies

Cancellations, negative quantities, zero prices, and negative prices require business-aware interpretation. Do not automatically delete these rows.

In [4]:
df["invoiceno"] = df["invoiceno"].astype("string")
df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")

checks = {
    "Cancellation invoice rows": int(df["invoiceno"].str.startswith("C", na=False).sum()),
    "Negative quantity rows": int((df["quantity"] < 0).sum()),
    "Zero unit-price rows": int((df["unitprice"] == 0).sum()),
    "Negative unit-price rows": int((df["unitprice"] < 0).sum()),
    "Missing customer IDs": int(df["customerid"].isna().sum()),
    "Invalid invoice dates": int(df["invoicedate"].isna().sum()),
}
display(pd.Series(checks, name="count").to_frame())

,count
Cancellation invoice rows,9288
Negative quantity rows,10624
Zero unit-price rows,2515
Negative unit-price rows,2
Missing customer IDs,135080
Invalid invoice dates,0


## 5. Data quality decisions

- Retain the original dataset unchanged.
- Preserve transaction rows in the prepared dataset.
- Exclude unidentified customers only from analyses requiring customer identity.
- Apply cancellation and product filters explicitly for each analytical task.
- Treat net line value as an analytical measure, not necessarily recognized revenue or profit.